# PRACTICAL 6: Dataset Splitting & Balancing (SMOTE)

## Objective
Prepare the security log dataset for machine learning classification models:
1. **Train-Test Split**: Partition the dataset into an 80% training set and a 20% testing set using stratified sampling.
2. **Handle Class Imbalances**: Address the severe class imbalance (benign/normal traffic vs. sparse attacks like SQLi/path traversal) using Synthetic Minority Over-sampling Technique (SMOTE).
3. **Outputs**: Export `train_dataset.csv` and `test_dataset.csv`.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate
from sklearn.model_selection import train_test_split

plt.style.use('seaborn-v0_8')
%matplotlib inline

### 1. Load Feature-Engineered Dataset

In [ ]:
input_file = "engineered_features.csv"
print(f"Loading dataset from '{input_file}'...")
df = pd.read_csv(input_file, low_memory=False)
print(f"Total Records Loaded: {len(df):,}")

### 2. Perform 80/20 Stratified Split

In [ ]:
feature_cols = ['ip_request_count', 'time_diff_seconds', 'status_code_404_count', 'is_bot_agent', 'url_entropy']
df = df.dropna(subset=feature_cols + ['label']).reset_index(drop=True)

X = df[feature_cols]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Original Training Set Label Distribution:")
print(y_train.value_counts())

### 3. Balance the Dataset

In [ ]:
# To prevent memory exhaust on ~1.6M training records, we apply a hybrid strategy:
# Downsample majority classes to 50k, then oversample minority classes using SMOTE to match.
target_cap = 50000

try:
    from imblearn.over_sampling import SMOTE
    print("'imblearn' found. Applying hybrid Undersampling + SMOTE...")
    
    train_temp = X_train.copy()
    train_temp['label'] = y_train
    
    sampled_dfs = []
    for cls_name in train_temp['label'].unique():
        cls_df = train_temp[train_temp['label'] == cls_name]
        if len(cls_df) > target_cap:
            print(f"  • Downsampling majority class '{cls_name}' to {target_cap:,}...")
            sampled_dfs.append(cls_df.sample(n=target_cap, random_state=42))
        else:
            sampled_dfs.append(cls_df)
            
    downsampled_train = pd.concat(sampled_dfs).reset_index(drop=True)
    X_train_ds = downsampled_train[feature_cols]
    y_train_ds = downsampled_train['label']
    
    print(f"  • Applying SMOTE to oversample minority classes to {target_cap:,}...")
    smote = SMOTE(random_state=42)
    X_train_bal, y_train_bal = smote.fit_resample(X_train_ds, y_train_ds)
    
except ImportError:
    print("imblearn not installed. Falling back to Random Undersampling of majority class...")
    train_temp = X_train.copy()
    train_temp['label'] = y_train
    min_class_size = train_temp['label'].value_counts().min()
    
    balanced_dfs = []
    for cls_name in train_temp['label'].unique():
        cls_df = train_temp[train_temp['label'] == cls_name]
        balanced_dfs.append(cls_df.sample(n=min_class_size, random_state=42))
        
    balanced_train = pd.concat(balanced_dfs).sample(frac=1, random_state=42).reset_index(drop=True)
    X_train_bal = balanced_train[feature_cols]
    y_train_bal = balanced_train['label']

print("\nBalanced Training Set Label Distribution:")
print(y_train_bal.value_counts())

### 4. Visualize Distribution & Save Outputs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Before balancing
sns.countplot(x=y_train, ax=axes[0], order=y_train.value_counts().index, hue=y_train, palette="viridis", legend=False)
axes[0].set_title("Before Balancing (Original Train Set)", fontsize=12, fontweight='bold')
axes[0].set_yscale('log')
axes[0].set_ylabel("Count (Log Scale)")
axes[0].tick_params(axis='x', rotation=30)

# After balancing
sns.countplot(x=y_train_bal, ax=axes[1], order=y_train_bal.value_counts().index, hue=y_train_bal, palette="viridis", legend=False)
axes[1].set_title("After Balancing (SMOTE / Resampled Train Set)", fontsize=12, fontweight='bold')
axes[1].set_ylabel("Count")
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig("../plots/plot_p6_class_distribution.png", dpi=300)
plt.show()

train_df_out = X_train_bal.copy()
train_df_out['label'] = y_train_bal
train_df_out.to_csv("train_dataset.csv", index=False)

test_df_out = X_test.copy()
test_df_out['label'] = y_test
test_df_out.to_csv("test_dataset.csv", index=False)
print("Successfully saved 'train_dataset.csv' and 'test_dataset.csv'!")